### Statistiche sul grafo completo

In [23]:
from step2_costruzione_grafo import Graph 

In [24]:
grafo = Graph.load_graph("/code/ADSproject/data/grafo.pkl")
print(f"nodi: {len(grafo.get_nodes())}")
print(f"archi: {len(grafo.get_edges())}")

nodi: 38252
archi: 115702


In [25]:
largest_component = Graph.load_graph("/code/ADSproject/data/grafo_largest_component.pkl")
print(f"nodi: {len(grafo.get_nodes())}")
print(f"archi: {len(grafo.get_edges())}")

# quindi la componente connessa più grande è il grafo stesso, top

nodi: 38252
archi: 115702


In [26]:
# conferma 

def conta_componenti_connesse(graph):
    visitati = set()
    componenti = []
    
    for nodo in graph.get_nodes():
        if nodo not in visitati:
            # nuova componente — fai DFS
            componente = set()
            stack = [nodo]
            while stack:
                n = stack.pop()
                if n not in visitati:
                    visitati.add(n)
                    componente.add(n)
                    for vicino in graph.adjacency_list[n]:
                        if vicino not in visitati:
                            stack.append(vicino)
            componenti.append(componente)
    
    return componenti

componenti = conta_componenti_connesse(grafo)
print(f"numero componenti connesse: {len(componenti)}")
print(f"dimensione componente più grande: {max(len(c) for c in componenti)}")
print(f"dimensione componente più piccola: {min(len(c) for c in componenti)}")

numero componenti connesse: 1
dimensione componente più grande: 38252
dimensione componente più piccola: 38252


In [27]:
## Confrontiamo il grafo nostro con quello as-rel 

import bz2

# leggo archi da as-rel
archi_as_rel = set()
with bz2.open("/code/ADSproject/data/20110501.as-rel.txt.bz2", "rt") as f:
    for riga in f:
        if riga.startswith("#"):
            continue
        parti = riga.strip().split("|")
        if len(parti) >= 2:
            u = int(parti[0])
            v = int(parti[1])
            archi_as_rel.add((min(u,v), max(u,v))) # teniamo solo in una direzione

In [28]:
# leggo archi dal mio grafo
archi_grafo = set()
for u, v, w in grafo.get_edges():
    archi_grafo.add((min(u,v), max(u,v)))

print(f"archi in as-rel: {len(archi_as_rel)}")
print(f"archi nel grafo: {len(archi_grafo)}")
print(f"archi in comune: {len(archi_grafo & archi_as_rel)}")
print(f"archi nel grafo ma non in as-rel: {len(archi_grafo - archi_as_rel)}")
print(f"archi in as-rel ma non nel grafo: {len(archi_as_rel - archi_grafo)}")

archi in as-rel: 112728
archi nel grafo: 115702
archi in comune: 109683
archi nel grafo ma non in as-rel: 6019
archi in as-rel ma non nel grafo: 3045


### Test per la complessità (a volte faccio dei rerun di verifica quindi i risultati cambiano leggermente - ricordati di committare la def e cambiare i numeri)

In [29]:
# importiamo le funzioni principali da testare direttamente dai moduli di step1/step2/step3
from step1_parser_cammini import leggi_cammini
from step3_ricerca_cammini_minimax import kruskal, minimax_query_dfs
from time import perf_counter
import pandas as pd
import random

BZ2_PATH = "/code/ADSproject/data/20110501.all-paths.bz2"
PKL_1M_PATH = "/code/ADSproject/data/test_data/cammini_test.pkl"  # 1M di cammini già estratti in step1, usato solo per il confronto bz2 vs pkl

# dimensioni crescenti dell'input, usate in tutte le celle sotto
# più avanti aggiungiamo anche il grafo completo con tutti i cammini del dataset
SIZES = [5_000, 10_000, 25_000, 50_000, 100_000, 250_000, 500_000, 1_000_000, 2_000_000]

random.seed(42)  # fissa il seed per rendere riproducibili le query casuali usate più avanti

#### Step 1: Parsing dei cammini - funzione `leggi_cammini` — atteso O(N)

Leggiamo un numero crescente di cammini dal file bz2 e misuriamo il tempo. Se il tempo cresce all'incirca proporzionalmente a N (N raddoppia → tempo raddoppia), la complessità lineare dichiarata nel README è confermata.

In [30]:
righe = []
for n in SIZES:
    t0 = perf_counter()
    cammini = leggi_cammini(BZ2_PATH, max_righe=n)  # rilegge il bz2 dall'inizio ogni volta, fino a n cammini validi
    t1 = perf_counter()
    righe.append({"N cammini letti": len(cammini), "tempo (s)": round(t1 - t0, 4)})

tabella_step1 = pd.DataFrame(righe)
tabella_step1  

,N cammini letti,tempo (s)
0,5000,0.3457
1,10000,0.0324
2,25000,0.0796
3,50000,0.1509
4,100000,0.3087
5,250000,0.7658
6,500000,1.9784
7,1000000,3.7694
8,2000000,7.6670


**Lettura del risultato:** guardando le ultime due righe, N raddoppia (1.000.000 → 2.000.000 cammini) e il tempo passa da 4.06s a 7.40s, cioè quasi raddoppia anche lui (×1.82). Sulle dimensioni piccole (es. 5.000 → 10.000) il tempo non raddoppia altrettanto, probabilmente c'è un costo fisso di apertura/decompressione del file bz2 che pesa di più quando il lavoro vero e proprio è ancora poco. Il trend generale possiamo dire che resta comunque lineare.

### Step 2: `build_from_bz2` — atteso O(N)

Questa volta costruiamo il grafo (parsing + aggiornamento frequenze). Conserviamo i grafi costruiti nel dizionario `grafi_per_dimensione` per riusarli nelle celle successive (componente connessa, Kruskal, query).

In [31]:
grafi_per_dimensione = {}  # salviamo qui i grafi costruiti a ogni dimensione
righe = []
for n in SIZES:
    g = Graph(directed=False)
    t0 = perf_counter()
    g.build_from_bz2(BZ2_PATH, max_paths=n)  # parsing + costruzione del grafo in un solo passaggio (non passa dai pkl di step1)
    t1 = perf_counter()
    grafi_per_dimensione[n] = g
    righe.append({
        "N cammini processati": n,
        "nodi": len(g.get_nodes()),
        "archi": len(g.get_edges()),
        "tempo (s)": round(t1 - t0, 4),
    })

tabella_step2 = pd.DataFrame(righe)
tabella_step2

cammini letti: 5000
cammini letti: 10000
cammini letti: 25000
cammini letti: 50000
cammini letti: 100000
cammini letti: 250000
cammini letti: 500000
cammini letti: 1000000
cammini letti: 2000000


,N cammini processati,nodi,archi,tempo (s)
0,5000,3510,4930,0.0410
1,10000,5625,7933,0.0687
2,25000,9830,14191,0.1653
3,50000,14405,21323,0.3336
4,100000,19949,30526,0.6581
5,250000,28416,45513,1.7197
6,500000,33927,56952,3.3740
7,1000000,37020,65910,6.8994
8,2000000,37716,70962,13.8050


**Lettura del risultato:** N raddoppia (1M → 2M) e il tempo va da 6.82s a 13.43s (×1.97, quasi esattamente il doppio) — coerente con O(N). Da notare che nodi e archi non raddoppiano insieme a N (37.020 → 37.716 nodi, solo +2%): il grafo delle Autonomous System è quasi tutto già "scoperto" con 1M di cammini, quelli aggiuntivi per lo più aggiornano le frequenze di archi già esistenti invece di crearne di nuovi. Il tempo però continua a crescere linearmente lo stesso, perché `build_from_bz2` processa ogni cammino letto indipendentemente dal fatto che introduca qualcosa di nuovo nel grafo o no.

### Step 2: confronto bz2 vs pkl (a parità di dati, N = 1.000.000 cammini)

Nel README ipotizziamo che leggere da un pkl già estratto sia più veloce, perché il parsing del testo è già stato fatto in step1. Confrontiamo: costruzione diretta dal bz2 vs caricamento del pkl + costruzione del grafo.

In [32]:
# stessa quantità di dati (1.000.000 di cammini), due modalità di lettura a confronto

t0 = perf_counter()
g_bz2 = Graph(directed=False)
g_bz2.build_from_bz2(BZ2_PATH, max_paths=1_000_000)  # legge il testo grezzo dal bz2 e fa il parsing al volo
tempo_bz2 = perf_counter() - t0

t0 = perf_counter()
cammini_pkl = Graph.load_paths(filepath_pkl=PKL_1M_PATH)  # carica cammini già estratti e "puliti" da step1 
tempo_carico_pkl = perf_counter() - t0

t0 = perf_counter()
g_pkl = Graph(directed=False)
g_pkl.build_from_paths(cammini_pkl)  # stessa logica di costruzione di build_from_bz2, ma partendo da cammini già pronti
tempo_costruisci_pkl = perf_counter() - t0

tabella_confronto = pd.DataFrame([
    {"modalità": "bz2 (parsing + costruzione grafo)", "tempo (s)": round(tempo_bz2, 4)},
    {"modalità": "pkl (caricamento pickle)", "tempo (s)": round(tempo_carico_pkl, 4)},
    {"modalità": "pkl (costruzione grafo)", "tempo (s)": round(tempo_costruisci_pkl, 4)},
    {"modalità": "pkl (totale)", "tempo (s)": round(tempo_carico_pkl + tempo_costruisci_pkl, 4)},  # da confrontare con la riga bz2 sopra
])
tabella_confronto

cammini letti: 1000000


,modalità,tempo (s)
0,bz2 (parsing + costruzione grafo),6.7777
1,pkl (caricamento pickle),0.9361
2,pkl (costruzione grafo),4.1421
3,pkl (totale),5.0782


**Lettura del risultato:** il totale pkl (caricamento + costruzione grafo) è 4.78s contro i 7.08s del bz2 diretto — il pkl è circa 1.5× più veloce. Scomponendo i tempi si vede perché: caricare il pkl costa solo 0.67s. La differenza tra 7.08s e 4.78s sta quindi tutta nel parsing del testo grezzo (split delle righe, rimozione degli indirizzi IP, ecc.), che nel flusso pkl è già stato fatto una volta per tutte in step1. Questo conferma l'ipotesi.

### Step 2: `largest_connected_component` — atteso O(V+E)

Misuriamo il tempo della DFS che trova la componente connessa più grande, sui grafi costruiti sopra più il grafo completo già caricato all'inizio del notebook (variabile `grafo`). Conserviamo anche le componenti connesse (sottografi) per usarle con Kruskal.

In [33]:
grafi = dict(grafi_per_dimensione)  # copia il dizionario per non modificare l'originale
grafi["completo"] = grafo  # aggiungiamo anche il grafo con tutti i cammini del dataset (caricato a inizio notebook)

righe = []
sottografi = {}
for chiave, g in grafi.items():
    v = len(g.get_nodes())
    e = len(g.get_edges())
    t0 = perf_counter()
    g.largest_connected_component()  # DFS iterativa: qui ci interessa solo il tempo, non serve salvare i nodi restituiti
    t1 = perf_counter()
    sottografi[chiave] = g.get_largest_connected_subgraph()  # rifà la DFS internamente e restituisce un Graph vero e proprio: ci serve per Kruskal
    righe.append({
        "dimensione": chiave,
        "V": v,
        "E": e,
        "V+E": v + e,
        "tempo (s)": round(t1 - t0, 5),
    })

tabella_componente = pd.DataFrame(righe)
tabella_componente

,dimensione,V,E,V+E,tempo (s)
0,5000,3510,4930,8440,0.00159
1,10000,5625,7933,13558,0.00246
2,25000,9830,14191,24021,0.00414
3,50000,14405,21323,35728,0.00681
4,100000,19949,30526,50475,0.01112
5,250000,28416,45513,73929,0.01682
6,500000,33927,56952,90879,0.02209
7,1000000,37020,65910,102930,0.02645
8,2000000,37716,70962,108678,0.02692
9,completo,38252,115702,153954,0.03587


**Lettura del risultato:** (V+E) cresce da 8.440 a 153.954 (×18) e il tempo cresce da 0.0016s a 0.037s (×23) — stesso ordine di grandezza, coerente con O(V+E).

### Step 3: `kruskal` — atteso O(E log V)

Costruiamo l'MST con Kruskal su ogni componente connessa trovata sopra, e misuriamo il tempo. In questo dataset V cambia poco tra le dimensioni testate (quindi log₂(V) resta quasi costante), quindi ci aspettiamo che il tempo cresca quasi proporzionalmente a E.

In [34]:
righe = []
mst_per_dimensione = {}
for chiave, sg in sottografi.items():  # sg è già la componente connessa più grande
    v = len(sg.get_nodes())
    e = len(sg.get_edges())
    if v < 2:
        continue  # kruskal ha senso solo con almeno 2 nodi
    t0 = perf_counter()
    mst, peso_totale = kruskal(sg)  # ordina gli archi per peso e li aggiunge se non creano cicli (Union-Find con path compression + union by rank)
    t1 = perf_counter()
    mst_per_dimensione[chiave] = mst  # ci serve nella prossima cella per le query minimax
    righe.append({
        "dimensione": chiave,
        "V": v,
        "E": e,
        "tempo (s)": round(t1 - t0, 5),
    })

tabella_kruskal = pd.DataFrame(righe)
tabella_kruskal

,dimensione,V,E,tempo (s)
0,5000,3510,4930,0.01734
1,10000,5625,7933,0.01531
2,25000,9830,14191,0.02603
3,50000,14405,21323,0.04019
4,100000,19949,30526,0.05998
5,250000,28416,45513,0.09490
6,500000,33927,56952,0.12548
7,1000000,37020,65910,0.14648
8,2000000,37716,70962,0.16064
9,completo,38252,115702,0.26222


**Lettura del risultato:** il numero di archi (E) cresce da 4.930 a 115.702, cioè di circa 23 volte, mentre il tempo di esecuzione cresce da 0,00885 s a 0,30938 s, cioè di circa 35 volte.
Questo andamento è coerente con la complessità teorica di Kruskal, pari a (O(E\log E (o V))). Il tempo non cresce quindi in modo perfettamente lineare con (E), perché oltre al numero di archi conta anche il costo del loro ordinamento, rappresentato dal fattore (\log E). Poiché il logaritmo cresce lentamente, il numero di archi resta il fattore principale. Le piccole irregolarità nei tempi possono dipendere dal rumore dell’ambiente di esecuzione.


### Step 3: `minimax_query_dfs` — atteso O(V)

Per ogni MST costruito sopra, eseguiamo 200 query minimax tra coppie di nodi casuali e misuriamo il tempo medio per query.

In [35]:
N_QUERY = 200  # numero di query casuali per non dipendere da una singola coppia (start, target) che potrebbe esserci o no 
righe = []
for chiave, mst in mst_per_dimensione.items():
    nodi = mst.get_nodes()
    v = len(nodi)
    if v < 2:
        continue
    tempi = []
    for _ in range(N_QUERY):
        start, target = random.sample(nodi, 2)  # coppia di nodi casuali distinti
        t0 = perf_counter()
        minimax_query_dfs(mst, start, target)  # DFS sull'MST: tra due nodi esiste un solo cammino possibile, il costo è il massimo peso incontrato
        t1 = perf_counter()
        tempi.append(t1 - t0)
    righe.append({
        "dimensione": chiave,
        "V": v,
        "tempo medio per query (ms)": round(1000 * sum(tempi) / len(tempi), 4),
    })

tabella_query = pd.DataFrame(righe)
tabella_query

,dimensione,V,tempo medio per query (ms)
0,5000,3510,0.8941
1,10000,5625,1.6247
2,25000,9830,2.6591
3,50000,14405,3.9171
4,100000,19949,5.8832
5,250000,28416,8.3848
6,500000,33927,11.7925
7,1000000,37020,12.9172
8,2000000,37716,13.6263
9,completo,38252,13.7241


**Lettura del risultato:** il numero di vertici (V) cresce da 3.510 a 38.252, aumentando di circa 11 volte, mentre il tempo medio per query passa da 0,9863 ms a 14,4183 ms, aumentando di circa 15 volte.
Il risultato è compatibile con una complessità (O(V)). La DFS può infatti visitare, nel caso peggiore, tutti i vertici dell’albero prima di trovare il risultato o concludere che non esiste. Poiché la struttura visitata è un albero, il numero di archi è pari a (V-1), quindi la complessità generale (O(V+E)) si riduce a (O(V)). Per i valori più grandi, il tempo tende a stabilizzarsi perché anche il numero di vertici cresce molto poco: da 37.020 a 38.252.